# Open Images pilot — build a paired real/fake set and prove it before scaling

This is the **2,000-image stop** that `docs/02-open-weight-generators-on-open-images.md` §4
asks for, and the **encoder-parity gate** that `docs/03-commercial-apis-on-open-images.md` §3.1
puts ahead of spending any money.

It builds a small NTIRE-style paired corpus end to end:

| step | what happens |
|---|---|
| 1 | harvest CC BY 2.0 portrait photographs from Open Images V7 |
| 2 | pull each one's **Localized Narrative** — a human-written caption — as its prompt |
| 3 | generate a counterpart with an open-weight model (text-to-image, and optionally inpainting) |
| 4 | put every generated image through **encoder parity**, so it inherits its partner real's exact JPEG settings and pixel dimensions |
| 5 | **score the confound gate** and tell you pass or fail |

The output is a directory you can point `scripts/gate_confounds.py` at, plus a
straight answer to the one question blocking task 03: *does our save path leak
the label?*

### Why the pairing matters

This mirrors how the NTIRE 2026 challenge built its dataset (arXiv 2604.11487, §2.1).
They caption real images with a vision model and rewrite them into prompts, then note:

> *"By 'pairing' generated images with their real counterparts, we ensure that both
> subsets reflect similar semantics and content distribution, which should help
> detectors learn content-agnostic features."*

and, on the exact problem step 4 solves:

> *"To further minimize potential biases in generated imagery, we also align its
> distributions of resolutions, aspect ratios, JPEG compression quality factors,
> and other statistics to those of the real subset."*

They give no method for that alignment — one sentence is all the paper contains.
Step 4 is our version, and it is stronger than theirs: they match the two piles
*in distribution*, we match each fake to its own partner *exactly*.

Our captions are also better than theirs. Localized Narratives are written by
people looking at the photograph; NTIRE's are a vision model's guess.

### Before you start

1. **Settings → Accelerator → GPU T4 ×2** (or P100). On CPU step 3 will not finish.
2. **Settings → Internet → On.** Steps 1–3 all download.
3. Nothing here is gated and no token is needed. SDXL is CreativeML OpenRAIL++-M,
   which `docs/02` §2 clears for use.

Expect ~25 minutes at the default `N_REALS = 300`, most of it in step 3.

## 0. Parameters

`SMOKE = True` proves the whole chain on a handful of images in a few minutes.
Leave it on for your first run. Everything else has a working default.

In [ ]:
# ============ THE ONLY LINES YOU NORMALLY EDIT ============
SMOKE        = True     # True = 12 images, minutes. False = the real pilot.
N_REALS      = 300      # pairs to build when SMOKE is False. docs/02 §4 wants 2000 eventually.
GENERATOR    = "sdxl"   # SDXL 1.0 base. Do NOT add SDXL-Turbo: its licence tag is
                        # sai-nc-community (non-commercial), NOT the OpenRAIL++-M that
                        # docs/02 §2 lists. That row conflates 1.0 with Turbo and is wrong.
INPAINT_FRAC = 0.3      # docs/02 §3.1 wants 70/30 t2i/inpaint. 0.0 skips the second pipeline.
# ==========================================================

SEED       = 20260827          # matches scripts/extract_features.py
MAX_RATIO  = 0.7               # width/height ceiling, same as acquire_open_images_portrait.py
MIN_SHORT  = 400               # short side floor, same script
MAX_AUC    = 0.60              # docs/02 §5.2's own gate on jpeg_quality

WORK       = "/kaggle/working"
OUT        = f"{WORK}/ov7_pilot"
REALS_DIR  = f"{OUT}/portrait"          # <ImageID>.jpg, the authentic half
RAW_DIR    = f"{OUT}/raw_generated"     # what the generator emitted, before parity
FAKES_DIR  = f"{OUT}/generated_parity"  # after parity: this is the half you keep

if SMOKE:
    N_REALS, INPAINT_FRAC = 12, 0.0

# Open Images VALIDATION metadata: 15 MB and 41,620 rows, against 2.7 GB for the
# train split. Same columns, same licences, and far more than we need for a
# pilot -- there is no reason to pull the big one here.
META_URL = "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv"
NARR_URL = ("https://storage.googleapis.com/localized-narratives/annotations/"
            "open_images_validation_localized_narratives.jsonl")
CC_BY_2  = "https://creativecommons.org/licenses/by/2.0/"

for d in (OUT, REALS_DIR, RAW_DIR, FAKES_DIR):
    import os; os.makedirs(d, exist_ok=True)

print(f"SMOKE={SMOKE}  pairs={N_REALS}  generator={GENERATOR}  inpaint={INPAINT_FRAC:.0%}")

## 1. Get the code

Public repo, shallow clone, and a `reset --hard` on re-run so a resumed session
picks up any fix rather than a stale tree. Nothing here is authenticated.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection.git"
BRANCH   = "feat/encoder-parity-pilot"   # change to "master" once this merges
REPO_DIR = "/kaggle/working/robust-aigc-detection"

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

for p in (os.path.join(REPO_DIR, "src"), os.path.join(REPO_DIR, "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("repo at", REPO_DIR)

## 2. Install — without losing Kaggle's torch

The project goes in with `--no-deps`, which is a pure path registration. Handing
pip the `torch>=2.0` line from `pyproject.toml` invites it to resolve a torch
built for a different CUDA than this machine's drivers, and you get a torch that
cannot see the GPU with no way back except a factory reset.

`diffusers` and `accelerate` declare only a floor on torch, so they install
against the one already here rather than replacing it. The assert afterwards is
the thing that actually catches a broken environment.

In [ ]:
sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR])
sh([sys.executable, "-m", "pip", "install", "-q", "diffusers", "accelerate", "safetensors"])

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU. Step 3 will not finish on CPU."

from aigcdet.data.encoder_parity import save_matched_to_real, read_profile, ParityError
print("aigcdet imported OK")

## 3. The real half

Filtered exactly as `scripts/acquire_open_images_portrait.py` does it, and for the
same reasons:

* **`License == CC BY 2.0`** — the only vertical-real source audited that permits
  both commercial use *and* redistribution. Do not silently substitute Pexels or
  Unsplash; the first bars ML datasets outright and the second bars redistribution.
* **aspect ≤ 0.7, short side ≥ 400** — portrait, and big enough that a 200px crop
  is a crop rather than an upscale.

`attribution.csv` is written alongside, and it is not optional: CC BY requires
attribution, and normalisation strips image metadata, so that file is the only
surviving record of who took each photograph.

In [ ]:
import csv, io, urllib.request, threading
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

UA = "Mozilla/5.0 (compatible; aigcdet-research/1.0)"

def get(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def download(url, dst, min_bytes, label):
    """Download through `.part` and rename, the same convention as
    `normalize.save_png`. `open(dst,"wb")` truncates BEFORE the fetch runs, so a
    failed download otherwise leaves a 0-byte file that the resume check treats
    as done -- and you get a silent empty result on the next run instead of an
    error. The size floor catches a truncated body that still returned 200.
    """
    if os.path.exists(dst) and os.path.getsize(dst) >= min_bytes:
        print(f"{label} already present ({os.path.getsize(dst)/1e6:.1f} MB)")
        return dst
    print(f"downloading {label} ...")
    tmp = dst + ".part"
    try:
        blob = get(url, timeout=600)
        if len(blob) < min_bytes:
            raise OSError(f"{label} truncated: {len(blob)} bytes < {min_bytes}")
        with open(tmp, "wb") as fh:
            fh.write(blob)
        os.replace(tmp, dst)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise
    print(f"{label}: {os.path.getsize(dst)/1e6:.1f} MB")
    return dst

meta_path = download(META_URL, f"{OUT}/validation-images.csv", 10_000_000, "metadata")

with open(meta_path, newline="", encoding="utf-8") as fh:
    candidates = [r for r in csv.DictReader(fh)
                  if r.get("License") == CC_BY_2 and r.get("Thumbnail300KURL")]
assert candidates, "no CC BY 2.0 rows parsed -- delete the CSV and re-run"
print(f"{len(candidates):,} CC BY 2.0 rows with a thumbnail")

### Prompts, from the people who looked at the photograph

Localized Narratives are human-written descriptions keyed by `ImageID`. `docs/02`
§3.2 is explicit that we use these rather than inventing captions: they describe
the actual image, so the generated counterpart is a genuine pairing rather than
an unrelated photo.

The file is 1.1 GB, almost all of it per-word audio timings we do not want, so
this streams it and keeps only `caption` for IDs we might actually use — then
stops early once it has a comfortable surplus.

In [ ]:
import json

wanted = {r["ImageID"] for r in candidates}
# Surplus, and it has to be generous. Measured on this split: only ~4.9% of
# CC BY 2.0 candidates are portrait at <=0.7 with a short side >=400, and a
# further ~15% of thumbnail URLs are dead (Flickr rot). 25x leaves headroom on
# both. `acquire_open_images_portrait.py` measured 7.1% against the train split,
# so this one is slightly stingier.
need   = min(N_REALS * 25, len(wanted))
captions = {}

print(f"streaming narratives until {need:,} captions collected ...")
req = urllib.request.Request(NARR_URL, headers={"User-Agent": UA})
with urllib.request.urlopen(req, timeout=300) as r:
    for line in io.TextIOWrapper(r, encoding="utf-8"):
        try:
            rec = json.loads(line)
        except ValueError:
            continue
        iid = rec.get("image_id")
        if iid in wanted and iid not in captions:
            cap = (rec.get("caption") or "").strip()
            if len(cap) > 20:                      # a usable prompt, not a fragment
                captions[iid] = cap
                if len(captions) >= need:
                    break

assert captions, "no captions matched -- the narratives stream returned nothing usable"
print(f"{len(captions):,} captions")
print("example:", next(iter(captions.values()))[:160])

### Fetch the thumbnails

**These are thumbnails, not originals, and that is the whole reason step 5 exists.**
A `Thumbnail300KURL` is a *re-encoded JPEG*, and this project has already measured
that JPEG history leaks the label (`docs/low_level_confounds.md`: `jpeg_quality`
AUC 0.5532). Generated images arrive as clean PNGs with no compression history at
all, so stored side by side, "has JPEG artefacts" answers "is it real" perfectly.

Nothing is wrong with using thumbnails — 60k originals would be ~180 GB. It just
means parity is mandatory rather than nice to have.

In [ ]:
lock, kept, seen, fail = threading.Lock(), [], 0, 0
rows = [r for r in candidates if r["ImageID"] in captions]

def fetch(row):
    global seen, fail
    with lock:
        if len(kept) >= N_REALS:
            return
        seen += 1
    try:
        blob = get(row["Thumbnail300KURL"], timeout=25)
        with Image.open(io.BytesIO(blob)) as im:
            w, h = im.size
    except Exception:
        with lock: fail += 1
        return
    # Portrait, and big enough that a 200px crop is a crop and not an upscale.
    if w >= h or (w / h) > MAX_RATIO or min(w, h) < MIN_SHORT:
        return
    iid = row["ImageID"]
    with lock:
        if len(kept) >= N_REALS:
            return
        open(f"{REALS_DIR}/{iid}.jpg", "wb").write(blob)
        kept.append({"ImageID": iid, "size": f"{w}x{h}", "ratio": round(w / h, 4),
                     "Author": row.get("Author", ""),
                     "AuthorProfileURL": row.get("AuthorProfileURL", ""),
                     "Title": row.get("Title", ""),
                     "OriginalURL": row.get("OriginalURL", ""),
                     "License": CC_BY_2, "caption": captions[iid]})

with ThreadPoolExecutor(max_workers=32) as ex:
    list(ex.map(fetch, rows))

assert kept, (f"nothing survived the filter over {seen} candidates. Loosen "
              f"MAX_RATIO ({MAX_RATIO}) or MIN_SHORT ({MIN_SHORT}), or raise `need`.")

with open(f"{OUT}/attribution.csv", "w", newline="", encoding="utf-8") as fh:
    wtr = csv.DictWriter(fh, fieldnames=list(kept[0].keys()))
    wtr.writeheader(); wtr.writerows(kept)

sides = sorted(min(*(int(x) for x in k["size"].split("x"))) for k in kept)
print(f"kept {len(kept)}  checked {seen}  dead {fail}  yield {len(kept)/max(seen,1):.1%}")
print(f"real short side: min {sides[0]}  median {sides[len(sides)//2]}  max {sides[-1]}")
print(f"attribution.csv written ({len(kept)} rows)")

if len(kept) < N_REALS:
    print(f"\n  NOTE: asked for {N_REALS}, got {len(kept)}. Everything below still "
          f"works on {len(kept)} pairs.\n  The validation split holds ~40k CC BY 2.0 "
          f"rows and this filter keeps ~5%, so it tops out near 1,900 pairs.\n"
          f"  For docs/02 §4's full 2,000+, switch META_URL to the train split:\n"
          f"    https://storage.googleapis.com/openimages/v6/oidv6-train-images-with-labels-with-rotation.csv\n"
          f"  Same columns, 2.7 GB instead of 15 MB, and ~1.7M rows to draw from.")

# The generated side must clear this or parity will refuse the pair (no upscaling).
print(f"\n  generation resolution must have short side >= {sides[-1]}")

## 4. Generate the counterparts

**Generate taller than the reals, on purpose.** Parity resizes each fake down onto
its partner's exact dimensions and *refuses to upscale* — inventing detail would
fabricate the forensic evidence this corpus exists to measure. So the generated
short side has to clear the largest real short side printed above.

This is also a live finding for task 03: asking a commercial API for a square
1024×1024 and cropping to portrait leaves only ~614px on the short side, and in
testing that lost **19% of pairs**. Ask the API for portrait output natively
instead. At ~$0.05 an image that is real money for a one-line change.

`sdxl` runs at 832×1216, a native SDXL portrait bucket.

**SDXL-Turbo is not offered here, on purpose.** Its Hub licence tag is
`sai-nc-community` — non-commercial, with commercial use behind a Stability
membership. `docs/02` §2 lists "SDXL 1.0 / SDXL-Turbo | CreativeML OpenRAIL++-M |
Use", which is correct for 1.0 and wrong for Turbo. That row still needs fixing.

In [ ]:
import gc, torch
from diffusers import AutoPipelineForText2Image

# SDXL-Turbo is deliberately absent. Its Hub licence tag is `sai-nc-community`
# and commercial use needs a Stability membership, so it fails docs/02 §2's
# "check the weight licence, every time" rule. docs/02's table pairs it with
# SDXL 1.0 under OpenRAIL++-M, which is right for 1.0 and wrong for Turbo.
MODELS = {
    "sdxl": ("stabilityai/stable-diffusion-xl-base-1.0", (832, 1216), 25, 5.0),
}
model_id, (GW, GH), STEPS, GUID = MODELS[GENERATOR]
print(f"{model_id}  ->  {GW}x{GH}, {STEPS} steps")

pipe = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

n_t2i = len(kept) - int(len(kept) * INPAINT_FRAC)
t2i_rows, inpaint_rows = kept[:n_t2i], kept[n_t2i:]
print(f"{len(t2i_rows)} text-to-image, {len(inpaint_rows)} inpaint")

In [ ]:
import time
t0 = time.time()
for i, row in enumerate(t2i_rows):
    dst = f"{RAW_DIR}/{row['ImageID']}.png"
    if os.path.exists(dst):
        continue
    g = torch.Generator("cuda").manual_seed(SEED + i)
    img = pipe(prompt=row["caption"], width=GW, height=GH,
               num_inference_steps=STEPS, guidance_scale=GUID, generator=g).images[0]
    img.save(dst)
    if (i + 1) % 25 == 0 or i + 1 == len(t2i_rows):
        el = time.time() - t0
        print(f"  {i+1}/{len(t2i_rows)}  {el:.0f}s  ({el/(i+1):.1f}s/image)", flush=True)

del pipe; gc.collect(); torch.cuda.empty_cache()
print(f"text-to-image done in {time.time()-t0:.0f}s")

### The inpainted 30%

`docs/02` §3.1: partially-synthetic images are *a different detection problem* from
fully-synthetic ones, and the one we have almost no coverage of. Most authentic
pixels stay in place, so most forensic cues stay authentic too.

The mask is a centred box over roughly a third of the frame. Deliberately crude —
the point of the pilot is to prove the chain, and a smarter mask (segment the
subject, inpaint the background) is a decision for the real build, not this one.

In [ ]:
if inpaint_rows:
    from diffusers import AutoPipelineForInpainting
    from PIL import ImageDraw

    ipipe = AutoPipelineForInpainting.from_pretrained(
        "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
    ipipe.set_progress_bar_config(disable=True)

    def centre_mask(w, h, frac=0.34):
        m = Image.new("L", (w, h), 0)
        mw, mh = int(w * frac ** 0.5), int(h * frac ** 0.5)
        ImageDraw.Draw(m).rectangle(
            [(w - mw) // 2, (h - mh) // 2, (w + mw) // 2, (h + mh) // 2], fill=255)
        return m

    t0 = time.time()
    for i, row in enumerate(inpaint_rows):
        dst = f"{RAW_DIR}/{row['ImageID']}.png"
        if os.path.exists(dst):
            continue
        # Inpaint over the REAL, upscaled to the generation bucket, so the output
        # clears parity's no-upscale rule the same way the t2i half does.
        with Image.open(f"{REALS_DIR}/{row['ImageID']}.jpg") as im:
            base = im.convert("RGB").resize((GW, GH), Image.LANCZOS)
        g = torch.Generator("cuda").manual_seed(SEED + 10_000 + i)
        img = ipipe(prompt=row["caption"], image=base, mask_image=centre_mask(GW, GH),
                    width=GW, height=GH, num_inference_steps=max(STEPS, 20),
                    guidance_scale=max(GUID, 7.0), generator=g).images[0]
        img.save(dst)
        if (i + 1) % 25 == 0 or i + 1 == len(inpaint_rows):
            print(f"  {i+1}/{len(inpaint_rows)}  {time.time()-t0:.0f}s", flush=True)

    del ipipe; gc.collect(); torch.cuda.empty_cache()
    print(f"inpainting done in {time.time()-t0:.0f}s")
else:
    print("INPAINT_FRAC = 0, skipping")

## 5. Encoder parity

For each generated image, read its partner real's **exact JPEG quantisation tables**,
subsampling, progressive flag and pixel dimensions, and re-encode the fake with all
of them.

Copying the tables rather than estimating a quality number is the point. Recovering
"quality 71" from a table means inverting the standard scaling, and that inversion is
lossy — the error would be systematic and correlated with the label, i.e. a small
version of the confound reintroduced inside the fix. Copying makes the two members
of a pair read *identically*, by construction rather than approximately.

Any skipped pair here means the generator emitted something smaller than its partner
real. Raise the generation resolution rather than letting parity upscale.

In [ ]:
from aigcdet.data.encoder_parity import save_matched_to_real, ParityError

done, skipped = 0, []
for row in kept:
    iid = row["ImageID"]
    src, dst = f"{RAW_DIR}/{iid}.png", f"{FAKES_DIR}/{iid}.jpg"
    if not os.path.exists(src):
        skipped.append((iid, "not generated")); continue
    try:
        with Image.open(src) as im:
            im.load()
            save_matched_to_real(im, dst, f"{REALS_DIR}/{iid}.jpg")
        done += 1
    except ParityError as e:
        skipped.append((iid, str(e)))
    except Exception as e:
        skipped.append((iid, f"{type(e).__name__}: {e}"))

print(f"parity applied to {done} images; {len(skipped)} skipped")
for iid, why in skipped[:5]:
    print("   ", iid, why)
if skipped:
    print("\n^ if these say 'would upscale', raise GW/GH above the max real short side.")

## 6. The gate

`scripts/prove_encoder_parity.py`, the same script and the same thresholds you would
run on the box. It scores each low-level proxy on the real half against the generated
half, before and after parity.

**How to read it.** These are AUCs: **0.5 means the statistic tells you nothing about
the label; 1.0 means it separates the two classes perfectly on its own.**

* `jpeg_quality (path-aware)` and `short_side` should land on **exactly 0.5000** —
  parity makes them identical by construction, so anything else means the pairing is
  broken, not that the confound is stubborn.
* `jpeg_quality (pixel-only)` is the gate. Task 02 §5.2 sets the bar at **0.60**.
* `laplacian_var` and `noise_floor` will *not* reach 0.5 and are not expected to —
  the generated content genuinely differs from the photograph. `gate_confounds.py`
  reads those against the frozen corpus baselines (0.6721 / 0.6374), not against 0.5.

> **On a `SMOKE = True` run, believe only the two exact 0.5000s.** Every other
> figure is computed over a dozen images, where an AUC's standard error is wider
> than the effect being measured — a "failing" 0.76 there means nothing at all.
> The two construction guarantees hold at any n, which is exactly what makes them
> the right thing to check first. Re-run with `SMOKE = False` before reading the
> gate as a verdict.

In [ ]:
rc = subprocess.run([sys.executable, f"{REPO_DIR}/scripts/prove_encoder_parity.py",
                     "--reals", REALS_DIR, "--generated", RAW_DIR,
                     "--out", f"{OUT}/gate_probe", "--n", str(len(kept)),
                     "--max-auc", str(MAX_AUC), "--workers", "4"],
                    cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"})
print("\nexit code:", rc.returncode, "(0 = pass, 1 = refused)")

## 7. What the answer means

**If it passed** — the save path is safe. You are clear to run this at
`SMOKE = False, N_REALS = 2000` for `docs/02` §4's real pilot, and clear to start
spending on task 03. Nothing about your reals has to change.

**If `jpeg_quality (pixel-only)` came back above 0.60** — post the number. The
residual is resampling history, not compression: your reals were shrunk to
thumbnails by Google's pipeline, the fakes by LANCZOS here, and different shrink
kernels leave different fine texture behind. Parity cannot copy a kernel it cannot
observe. The fix is to give *both* classes the same final downscale, which measured
0.69 → 0.60 on a synthetic stand-in — but it re-encodes the authentic half, so it is
a decision to take deliberately rather than a default.

**If `short_side` is not 0.5000** — the pairing is broken. Check that every file in
`generated_parity/` has a same-stem partner in `portrait/`.

### Where task 03 plugs in

Steps 1, 2, 5 and 6 are unchanged for commercial APIs. Only step 3 differs: instead
of a local pipeline, call the provider with `row["caption"]` as the prompt and write
the result to `RAW_DIR/<ImageID>.png`. Everything downstream already works.

Two things to carry across before you do:

* **Ask for portrait output natively.** Square-and-crop cost 19% of pairs in testing.
* **Do not put a fake and its own partner real in the same eval split.** NTIRE
  deliberately breaks the pairing for validation and test — *"we use only unique
  images without its paired counterpart to avoid potential advantage from selecting
  between multiple similar images."* Use the pairing to generate and to apply parity,
  then draw the eval set's real half from different photographs.

### Keeping the output

`/kaggle/working` is wiped when the session ends. To keep this: **File → Save Version**,
or publish `{OUT}` as a Kaggle Dataset. Per `docs/kaggle_fleet_runbook.md`, Datasets
are created private — leave it that way unless every licence in
`docs/dataset_licences.md` clears redistribution.